# M8 - Final Game Prediction

## Project: Blir spelet en hit? Förutspå framgång för Steam-spel

This notebook combines the classification model from Milestone 6 and the regression model from Milestone 7.

The purpose is to let Pixel Harbor enter information about a new Steam game and receive:

- The probability that the game becomes a hit.
- A recommendation: LOVANDE or REKOMMENDERAS INTE.
- An estimate of the number of positive reviews.

Both models use information that can be available before a game is released.

In [1]:
import pandas as pd
import numpy as np
import joblib
import sys

from pathlib import Path

# Find the project folder
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Make the preprocessing module available
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.insert(0, str(NOTEBOOK_DIR))

# Import the custom preprocessing module
import steam_preprocessing

# Load the classification model
classification_artifact = joblib.load(
    PROJECT_ROOT / "models" / "steam_hit_model.joblib"
)

hit_model = classification_artifact["model"]
hit_threshold = classification_artifact["threshold"]

# Load the review prediction model
review_artifact = joblib.load(
    PROJECT_ROOT / "models" / "reviews_model.joblib"
)

review_model = review_artifact["model"]
review_features = review_artifact["features"]

print("Both models loaded successfully!")

print("\nClassification model:")
print(classification_artifact["model_name"])
print("Threshold:", round(hit_threshold, 4))

print("\nReview prediction model:")
print(review_artifact["model_name"])
print("Features:", review_features)

Both models loaded successfully!

Classification model:
Logistic Regression
Threshold: 0.7873

Review prediction model:
Random Forest Regressor
Features: ['price', 'windows', 'mac', 'linux', 'required_age', 'categories_count', 'genres_count']


## Enter a New Game

The following example represents a fictional upcoming Steam game. Its information is entered as raw values, and the notebook prepares the required features for both prediction models.

In [2]:
# Example game information

new_game = {
    "price": 14.99,
    "required_age": 0,
    "windows": True,
    "mac": False,
    "linux": False,
    "release_month": 9,
    "categories": [
        "Single-player",
        "Steam Achievements",
        "Full controller support"
    ],
    "genres": [
        "Indie",
        "Adventure",
        "Casual"
    ],
    "short_description": (
        "Explore a mysterious world, solve puzzles "
        "and discover hidden secrets in this adventure game."
    )
}

print("New game information:")
for key, value in new_game.items():
    print(f"{key}: {value}")

New game information:
price: 14.99
required_age: 0
windows: True
mac: False
linux: False
release_month: 9
categories: ['Single-player', 'Steam Achievements', 'Full controller support']
genres: ['Indie', 'Adventure', 'Casual']
short_description: Explore a mysterious world, solve puzzles and discover hidden secrets in this adventure game.


In [7]:
# Prepare the input for the classification model

classification_input = pd.DataFrame([{
    "price": new_game["price"],
    "required_age": new_game["required_age"],
    "windows": new_game["windows"],
    "mac": new_game["mac"],
    "linux": new_game["linux"],
    "release_month": str(new_game["release_month"]),
    "categories": new_game["categories"],
    "genres": new_game["genres"],
    "short_description": new_game["short_description"],
    "categories_count": len(new_game["categories"]),
    "genres_count": len(new_game["genres"])
}])

# Prepare the input for the regression model

review_input = pd.DataFrame([{
    "price": new_game["price"],
    "windows": new_game["windows"],
    "mac": new_game["mac"],
    "linux": new_game["linux"],
    "required_age": new_game["required_age"],
    "categories_count": len(new_game["categories"]),
    "genres_count": len(new_game["genres"])
}])

review_input = review_input[review_features]

print("Classification input:")
display(classification_input)

print("\nReview prediction input:")
display(review_input)

Classification input:


,price,required_age,windows,mac,linux,release_month,categories,genres,short_description,categories_count,genres_count
0,14.99,0,True,False,False,9,"[Single-player, Steam Achievements, Full contr...","[Indie, Adventure, Casual]","Explore a mysterious world, solve puzzles and ...",3,3



Review prediction input:


,price,windows,mac,linux,required_age,categories_count,genres_count
0,14.99,True,False,False,0,3,3


In [8]:
# Predict the probability of becoming a hit

hit_probability = hit_model.predict_proba(
    classification_input
)[0][1]

# Apply the threshold selected in M6
hit_prediction = (
    "LOVANDE"
    if hit_probability >= hit_threshold
    else "REKOMMENDERAS INTE"
)

print("HIT PREDICTION")
print("-------------------------")
print(f"Hit probability: {hit_probability:.2%}")
print(f"Recommendation: {hit_prediction}")
print(f"Decision threshold: {hit_threshold:.4f}")

HIT PREDICTION
-------------------------
Hit probability: 52.66%
Recommendation: REKOMMENDERAS INTE
Decision threshold: 0.7873


In [9]:
# Predict positive reviews on the log scale
predicted_log_reviews = review_model.predict(review_input)[0]

# Convert back to the original review scale
predicted_reviews = np.expm1(predicted_log_reviews)

# Create an approximate range using the test MAE
review_mae = review_artifact["test_mae"]

lower_log = max(0, predicted_log_reviews - review_mae)
upper_log = predicted_log_reviews + review_mae

lower_reviews = np.expm1(lower_log)
upper_reviews = np.expm1(upper_log)

print("POSITIVE REVIEW PREDICTION")
print("-------------------------")
print(f"Estimated positive reviews: {predicted_reviews:,.0f}")
print(
    f"Approximate range: "
    f"{lower_reviews:,.0f} - {upper_reviews:,.0f}"
)
print(f"Test MAE (log scale): {review_mae:.4f}")

POSITIVE REVIEW PREDICTION
-------------------------
Estimated positive reviews: 18
Approximate range: 3 - 89
Test MAE (log scale): 1.5767


In [12]:
print("=" * 45)
print("       PIXEL HARBOR - GAME EVALUATION")
print("=" * 45)

print(f"\nEstimated hit probability: {hit_probability:.2%}")
print(f"Recommendation: {hit_prediction}")

print(f"\nEstimated positive reviews: {predicted_reviews:,.0f}")
print(
    f"Approximate range: "
    f"{lower_reviews:,.0f} - {upper_reviews:,.0f}"
)

print("\nLIKELY SIMILAR GAME TYPE")
print("-------------------------")
print("Game type:", " / ".join(new_game["genres"]))
print("Features:", ", ".join(new_game["categories"]))
print("(Based on the genres and categories entered.)")

print("\n" + "=" * 45)

       PIXEL HARBOR - GAME EVALUATION

Estimated hit probability: 52.66%
Recommendation: REKOMMENDERAS INTE

Estimated positive reviews: 18
Approximate range: 3 - 89

LIKELY SIMILAR GAME TYPE
-------------------------
Game type: Indie / Adventure / Casual
Features: Single-player, Steam Achievements, Full controller support
(Based on the genres and categories entered.)

